# FraudShield Advanced: Built-in Algorithms and Hyperparameter Optimization

## Concept Review: Algorithm Mode vs Script Mode for XGBoost

SageMaker provides two ways to use its built-in XGBoost algorithm:

- **Algorithm Mode**: You use the SageMaker-managed XGBoost container with **no custom
  training script**. You configure everything through hyperparameters passed to the
  `Estimator`. The container handles data loading (CSV or libsvm), training, and model
  serialization automatically. This mode is ideal when your data pipeline is
  straightforward and you do not need custom preprocessing inside the training job.

- **Script Mode**: You provide an `entry_point` Python script that defines `train()`,
  and optionally `model_fn()`, `input_fn()`, and `predict_fn()`. This gives you full
  control over the training loop, custom metrics, and preprocessing logic, while still
  running inside the managed XGBoost container.

In this lecture we use **Algorithm Mode** for simplicity, then layer on Hyperparameter
Optimization (HPO) to automatically search for the best configuration.

**Environment**: SageMaker Studio on `ml.t3.medium` | Free Tier training on `ml.m5.xlarge`

In [ ]:
%pip install -q sagemaker boto3 pandas numpy

In [ ]:
import time
import json
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
    CategoricalParameter,
)

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
prefix = "fraudshield-algorithms"

print(f"Role:    {role}")
print(f"Region:  {region}")
print(f"Bucket:  {bucket}")

---
## Stage 1: XGBoost in Algorithm Mode

### Concept Review: XGBoost Architecture

XGBoost (eXtreme Gradient Boosting) builds an ensemble of decision trees **sequentially**.
Each new tree is trained to correct the residual errors of the previous ensemble, using
gradient descent on a differentiable loss function:

- **Objective function** = Training loss + Regularization term.
  For binary classification we use `binary:logistic`, which optimizes log-loss and outputs
  probabilities.

- **Key hyperparameters**:
  - `num_round`: Number of boosting rounds (trees).
  - `max_depth`: Maximum depth per tree. Deeper trees capture more complex interactions
    but increase overfitting risk.
  - `eta` (learning rate): Shrinks each tree's contribution. Lower values require more
    rounds but often generalize better.
  - `subsample` / `colsample_bytree`: Row and column sampling per tree, acting as
    regularization.

SageMaker's built-in XGBoost is a distributed, optimized implementation that can scale
across multiple instances automatically.

In [ ]:
xgb_image_uri = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.5-1",
)
print(f"XGBoost container image: {xgb_image_uri}")

In [ ]:
xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/xgboost/output",
    sagemaker_session=sagemaker_session,
)

xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=100,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=10,
)

print("XGBoost Estimator configured (Algorithm Mode -- no entry_point).")
print(f"Instance type: ml.m5.xlarge (Free Tier eligible)")

In [ ]:
np.random.seed(42)
n_samples = 2000

amounts = np.random.exponential(200, n_samples)
hours = np.random.randint(0, 24, n_samples)
distances = np.random.exponential(100, n_samples)
merchant_risk = np.random.beta(2, 5, n_samples)
is_intl = np.random.binomial(1, 0.15, n_samples)
velocity = np.random.poisson(3, n_samples)
card_age = np.random.randint(1, 3650, n_samples)

fraud_score = (
    0.3 * (amounts > 500).astype(float)
    + 0.2 * ((hours < 5) | (hours > 22)).astype(float)
    + 0.2 * (distances > 300).astype(float)
    + 0.15 * (merchant_risk > 0.5).astype(float)
    + 0.1 * is_intl
    + 0.05 * (velocity > 6).astype(float)
)
is_fraud = (fraud_score > np.random.uniform(0, 1, n_samples) * 1.5).astype(int)

# Algorithm Mode CSV: target in first column, no header
train_df = pd.DataFrame({
    "is_fraud": is_fraud,
    "amount": np.round(amounts, 2),
    "hour": hours,
    "distance_from_home": np.round(distances, 1),
    "merchant_risk_score": np.round(merchant_risk, 4),
    "is_international": is_intl,
    "transaction_velocity_1h": velocity,
    "card_age_days": card_age,
})

split_idx = int(0.8 * n_samples)
train_data = train_df.iloc[:split_idx]
val_data = train_df.iloc[split_idx:]

train_data.to_csv("train.csv", index=False, header=False)
val_data.to_csv("validation.csv", index=False, header=False)

train_s3 = sagemaker_session.upload_data("train.csv", bucket=bucket, key_prefix=f"{prefix}/xgboost/data/train")
val_s3 = sagemaker_session.upload_data("validation.csv", bucket=bucket, key_prefix=f"{prefix}/xgboost/data/validation")

print(f"Train uploaded: {train_s3}")
print(f"Validation uploaded: {val_s3}")
print(f"Class distribution: {dict(zip(*np.unique(is_fraud, return_counts=True)))}")

In [ ]:
train_input = TrainingInput(train_s3, content_type="text/csv")
val_input = TrainingInput(val_s3, content_type="text/csv")

xgb_estimator.fit(
    inputs={"train": train_input, "validation": val_input},
    job_name=f"fraudshield-xgb-algo-{int(time.time())}",
    wait=True,
    logs="All",
)

print(f"\nTraining job: {xgb_estimator.latest_training_job.name}")
print(f"Model artifact: {xgb_estimator.model_data}")

---
## Stage 2: K-Means Clustering

### Concept Review: K-Means on SageMaker

K-Means is an **unsupervised** algorithm that partitions data points into `k` clusters
by minimizing within-cluster sum of squared distances. SageMaker's built-in implementation
uses a modified **web-scale k-means** approach:

- **Mini-batch processing**: Instead of iterating over the full dataset each step,
  it processes mini-batches, which is far more efficient for large datasets.
- **GPU-accelerated** option for very large datasets.

In FraudShield, K-Means can be used for **customer transaction segmentation** -- grouping
transactions by behavioral similarity to identify spending patterns, which can then serve
as features for downstream fraud models.

In [ ]:
kmeans_image_uri = image_uris.retrieve(
    framework="kmeans",
    region=region,
)
print(f"K-Means container image: {kmeans_image_uri}")

kmeans_estimator = Estimator(
    image_uri=kmeans_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/kmeans/output",
    sagemaker_session=sagemaker_session,
)

kmeans_estimator.set_hyperparameters(
    k=5,
    mini_batch_size=200,
    feature_dim=7,
    init_method="random",
    max_iterations=100,
    epochs=10,
)

print("K-Means Estimator configured: k=5 clusters, 7 features")

In [ ]:
# K-Means: feature-only data, no target column
feature_cols = ["amount", "hour", "distance_from_home", "merchant_risk_score",
                "is_international", "transaction_velocity_1h", "card_age_days"]

kmeans_data = train_df[feature_cols].copy()

# K-Means built-in expects headerless CSV or RecordIO-protobuf
kmeans_data.to_csv("kmeans_train.csv", index=False, header=False)
kmeans_s3 = sagemaker_session.upload_data(
    "kmeans_train.csv", bucket=bucket, key_prefix=f"{prefix}/kmeans/data"
)

print(f"K-Means data uploaded: {kmeans_s3}")
print(f"Shape: {kmeans_data.shape}")

In [ ]:
kmeans_input = TrainingInput(kmeans_s3, content_type="text/csv")

kmeans_estimator.fit(
    inputs={"train": kmeans_input},
    job_name=f"fraudshield-kmeans-{int(time.time())}",
    wait=True,
    logs="All",
)

print(f"\nK-Means training job: {kmeans_estimator.latest_training_job.name}")
print(f"Model artifact: {kmeans_estimator.model_data}")

In [ ]:
# Deploy a temporary endpoint to inspect cluster assignments
kmeans_predictor = kmeans_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
)

sample_features = kmeans_data.head(10).values
predictions = kmeans_predictor.predict(sample_features)

print("K-Means cluster assignments for 10 sample transactions:")
for i, pred in enumerate(predictions["predictions"]):
    cluster = int(pred["closest_cluster"])
    distance = round(pred["distance_to_cluster"], 4)
    print(f"  Transaction {i}: Cluster {cluster} (distance: {distance})")

# Clean up the endpoint immediately
kmeans_predictor.delete_endpoint()
print("\nK-Means endpoint deleted.")

---
## Stage 3: Hyperparameter Optimization (HPO)

### Concept Review: HPO Job Anatomy

An HPO (tuning) job launches multiple training jobs, each with a different hyperparameter
configuration, and selects the best one based on an objective metric.

**Strategies**:

- **Bayesian Optimization** (default): Builds a probabilistic **surrogate model** of the
  objective function. Each completed trial updates the surrogate, which guides the next
  trial toward promising regions of the search space. This is sample-efficient but
  **inherently sequential** -- the surrogate improves most when each trial completes
  before the next starts.

- **Random Search**: Samples hyperparameters uniformly from the defined ranges. Trials
  are fully independent, so you can run them all in parallel. Less sample-efficient
  but useful when you have a large parallel budget.

**Parallelism trade-off**: Increasing `max_parallel_jobs` speeds up wall-clock time but
reduces Bayesian optimization's effectiveness because parallel trials cannot benefit from
each other's results. A common practice is to set `max_parallel_jobs` to 2-3 for Bayesian
and to `max_jobs` for Random.

In [ ]:
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
)

print("Imported HPO classes: HyperparameterTuner, ContinuousParameter, IntegerParameter")

In [ ]:
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.01, 0.3),
    "max_depth": IntegerParameter(3, 10),
    "min_child_weight": IntegerParameter(1, 10),
    "subsample": ContinuousParameter(0.5, 1.0),
    "colsample_bytree": ContinuousParameter(0.5, 1.0),
    "num_round": IntegerParameter(50, 300),
}

print("Hyperparameter search ranges:")
for name, param_range in hyperparameter_ranges.items():
    print(f"  {name}: {param_range}")

In [ ]:
# Create a fresh base estimator for the tuner (Algorithm Mode)
hpo_base_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/hpo/output",
    sagemaker_session=sagemaker_session,
)

hpo_base_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=10,
)

tuner = HyperparameterTuner(
    estimator=hpo_base_estimator,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type="Maximize",
    max_jobs=10,
    max_parallel_jobs=2,
    strategy="Bayesian",
)

print("HyperparameterTuner configured:")
print(f"  Strategy:         Bayesian")
print(f"  Objective:        Maximize validation:auc")
print(f"  Max jobs:         10")
print(f"  Max parallel:     2")
print(f"  Instance type:    ml.m5.xlarge")

In [ ]:
tuner.fit(
    inputs={"train": train_input, "validation": val_input},
    job_name=f"fraudshield-hpo-{int(time.time())}",
    wait=False,
)

print(f"HPO job launched: {tuner.latest_tuning_job.name}")
print("Training in background -- monitor in the SageMaker console under")
print("Hyperparameter tuning jobs, or poll with the cells below.")

In [ ]:
sm = boto3.client("sagemaker")

tuning_job_name = tuner.latest_tuning_job.name

while True:
    resp = sm.describe_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=tuning_job_name
    )
    status = resp["HyperParameterTuningJobStatus"]
    counts = resp["TrainingJobStatusCounters"]
    completed = counts.get("Completed", 0)
    in_progress = counts.get("InProgress", 0)
    total = resp["HyperParameterTuningJobConfig"]["ResourceLimits"]["MaxNumberOfTrainingJobs"]
    print(f"  Status: {status} | Completed: {completed}/{total} | In-progress: {in_progress}")

    if status in ["Completed", "Failed", "Stopped"]:
        break
    time.sleep(60)

print(f"\nHPO job final status: {status}")

In [ ]:
best = resp["BestTrainingJob"]
best_job_name = best["TrainingJobName"]
best_metric = best["FinalHyperParameterTuningJobObjectiveMetric"]["Value"]

print(f"Best training job: {best_job_name}")
print(f"Best validation AUC: {best_metric}")

best_job_desc = sm.describe_training_job(TrainingJobName=best_job_name)
best_hps = best_job_desc["HyperParameters"]

print("\nBest hyperparameters:")
for hp_name in sorted(hyperparameter_ranges.keys()):
    print(f"  {hp_name:25s}  {best_hps.get(hp_name, 'N/A')}")

In [ ]:
tuning_analytics = tuner.analytics()
trials_df = tuning_analytics.dataframe()

print(f"Total trials: {len(trials_df)}")
print(f"\nTrials sorted by objective metric (descending):")

display_cols = ["TrainingJobName", "FinalObjectiveValue", "TrainingJobStatus"]
hp_cols = [c for c in trials_df.columns if c not in display_cols
           and c not in ["TrainingStartTime", "TrainingEndTime", "TrainingElapsedTimeSeconds"]]
display_cols.extend(sorted(hp_cols))

trials_sorted = trials_df.sort_values("FinalObjectiveValue", ascending=False)
display(trials_sorted[display_cols].head(10))

## Cleanup

Remove training artifacts from S3 and delete local temporary files.

In [ ]:
import os

# Delete S3 artifacts
s3 = boto3.resource("s3")
s3_bucket = s3.Bucket(bucket)

for sub_prefix in ["xgboost", "kmeans", "hpo"]:
    full_prefix = f"{prefix}/{sub_prefix}/"
    objects = list(s3_bucket.objects.filter(Prefix=full_prefix))
    if objects:
        s3_bucket.delete_objects(
            Delete={"Objects": [{"Key": obj.key} for obj in objects]}
        )
        print(f"Deleted {len(objects)} objects under s3://{bucket}/{full_prefix}")

# Remove local CSV files
for f in ["train.csv", "validation.csv", "kmeans_train.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed local file: {f}")

print("Cleanup complete.")